# K Nearest Neighbors (KNN) Algorithm

## Layman's Explanation

This is the most intuitive machine learning algorithm. You can understand this with a real life example. Suppose you are a class teacher in a school. You are on a school picnic. You see a student doing something wrong. You want to complaint to their class teacher. But you don't know which class they belong to. So you go and ask their nearby students with whom they are hanging out. You ask 5 (`K`) students and 4 of them say that the student is from 7th grade and 1 says that the student is from 8th grade. So you conclude that the student is from 7th grade. So you complaint to the 7th grade teacher. This is how KNN works. It looks at the K nearest neighbors and assigns the class that is most common among those neighbors.

## Theoretical Background

Consider a dataset with labeled examples. The KNN algorithm classifies a new example by finding the K most similar examples in the training set and assigning it the majority class among those neighbors.

### Notation
Let $X \in \mathbb{R}^{n \times d}$ be the matrix of training examples, where $n$ is the number of samples and $d$ is the number of features.
Let $y \in \{0, 1\}^n$ be the vector of labels for the training examples.
Let $x_{\text{new}} \in \mathbb{R}^d$ be the new example to classify.
Let $K \in \mathbb{N}$ be the number of nearest neighbors to consider.

### Distance Metrics
The similarity between two examples is measured using a distance metric. Common choices include but are not limited to:
*   **Minkowski Distance**: $\|x_i - x_j\|_p = (\sum_{k=1}^d |x_{ik} - x_{jk}|^p)^{1/p}$
*   **Euclidean Distance**: $\|x_i - x_j\|_2 = \sqrt{\sum_{k=1}^d (x_{ik} - x_{jk})^2}$
*   **Manhattan Distance**: $\|x_i - x_j\|_1 = \sum_{k=1}^d |x_{ik} - x_{jk}|$
*   **Canberra Distance**: $\|x_i - x_j\|_{\text{Canberra}} = \sum_{k=1}^d \frac{|x_{ik} - x_{jk}|}{|x_{ik}| + |x_{jk}|}$
*   **Chebyshev Distance**: $\|x_i - x_j\|_{\text{Chebyshev}} = \max_{k=1}^d |x_{ik} - x_{jk}|$
*   **Quadratic Distance**: $\|x_i - x_j\|_{\text{Quadratic}} = \sqrt{(x_i - x_j)^T S^{-1} (x_i - x_j)}$ where $S$ is the covariance matrix of the training data.
*   **Mahalanobis Distance**: $\|x_i - x_j\|_{\text{Mahalanobis}} = \sqrt{(x_i - x_j)^T S^{-1} (x_i - x_j)}$ where $S$ is the covariance matrix of the training data.
*   **Correlation Distance**: $\|x_i - x_j\|_{\text{Correlation}} = 1 - \frac{(x_i - \bar{x})^T (x_j - \bar{x})}{\|x_i - \bar{x}\|_2 \|x_j - \bar{x}\|_2}$ where $\bar{x}$ is the mean of the training data.
*   **Chi-Squared Distance**: $\|x_i - x_j\|_{\chi^2} = \sum_{k=1}^d \frac{(x_{ik} - x_{jk})^2}{x_{ik} + x_{jk}}$
*   **Kendall Distance**: $\|x_i - x_j\|_{\text{Kendall}} = \frac{2}{d(d-1)} \sum_{k < l} \text{sgn}(x_{ik} - x_{il}) \text{sgn}(x_{jk} - x_{jl})$

### Classification Process
For a new example $x_{\text{new}}$:
1. Compute the distance from $x_{\text{new}}$ to all training examples.
2. Select the K nearest neighbors based on these distances.
3. Assign the class label that appears most frequently among these K neighbors.

## Various Distance Metrics

The choice of distance metric can significantly affect the performance of the KNN algorithm.

In [1]:
import numpy as np
from collections import Counter

import numpy as np

def compute_distances(X, centroids, metric='euclidean', **kwargs):
    C = centroids
    eps = 1e-8 # Small epsilon to prevent division by zero

    if metric == 'minkowsky':
        r = kwargs.get('r', 3) # Minkowski requires parameter 'r' (default 3)
        return np.sum(np.abs(X[:, np.newaxis] - C)**r, axis=2)**(1/r)

    elif metric == 'euclidean':
        return np.sqrt(np.sum((X[:, np.newaxis] - C)**2, axis=2))

    elif metric == 'manhattan' or metric == 'city-block':
        return np.sum(np.abs(X[:, np.newaxis] - C), axis=2)

    elif metric == 'canberra':
        num = np.abs(X[:, np.newaxis] - C)
        den = np.abs(X[:, np.newaxis] + C) + eps
        return np.sum(num / den, axis=2)

    elif metric == 'chebychev':
        return np.max(np.abs(X[:, np.newaxis] - C), axis=2)

    elif metric == 'quadratic':
        # Requires a positive definite weight matrix Q (D x D)
        Q = kwargs.get('Q')
        if Q is None:
            Q = np.eye(X.shape[1]) # Fallback to identity matrix (Euclidean)

        diff = X[:, np.newaxis] - C # Shape: (N, K, D)
        # Vectorized (x-y)^T Q (x-y) using Einstein summation
        return np.einsum('nki,ij,nkj->nk', diff, Q, diff)

    elif metric == 'mahalanobis':
        # Requires covariance matrix V. If not provided, compute from X.
        V = kwargs.get('V')
        if V is None:
            V = np.cov(X.T) + np.eye(X.shape[1]) * eps

        V_inv = np.linalg.inv(V)
        m = X.shape[1]
        det_V = np.linalg.det(V)

        diff = X[:, np.newaxis] - C
        dist_sq = np.einsum('nki,ij,nkj->nk', diff, V_inv, diff)

        # Applying the specific [det V]^(1/m) formula from the image
        return (det_V**(1/m)) * dist_sq

    elif metric == 'correlation':
        # Center the data
        X_mean = X - X.mean(axis=1, keepdims=True)
        C_mean = C - C.mean(axis=1, keepdims=True)

        # Numerator: dot product of centered vectors
        num = np.dot(X_mean, C_mean.T)

        # Denominator: product of standard deviations
        den_X = np.sqrt(np.sum(X_mean**2, axis=1))[:, np.newaxis]
        den_C = np.sqrt(np.sum(C_mean**2, axis=1))[np.newaxis, :]

        correlation = num / (den_X * den_C + eps)

        # The image shows the correlation coefficient. To make it a
        # valid *distance* for K-Means, we subtract it from 1.
        return 1 - correlation

    elif metric == 'chi-square':
        # sum_i: sum of all values for attribute i in the training set
        sum_i = np.sum(X, axis=0) + eps

        # size_x, size_y: sum of all values in the respective vectors
        size_x = np.sum(X, axis=1, keepdims=True) + eps
        size_y = np.sum(C, axis=1, keepdims=True) + eps

        X_norm = X / size_x
        C_norm = C / size_y

        diff = X_norm[:, np.newaxis] - C_norm # Shape: (N, K, D)
        return np.sum((diff**2) / sum_i, axis=2)

    elif metric == 'kendall':
        m = X.shape[1]
        if m < 2:
            return np.zeros((X.shape[0], C.shape[0]))

        # Get indices for all unique feature pairs (i, j) where j < i
        i_idx, j_idx = np.tril_indices(m, k=-1)

        # Compute signs of differences for all pairs
        sign_X = np.sign(X[:, i_idx] - X[:, j_idx]) # Shape: (N, num_pairs)
        sign_C = np.sign(C[:, i_idx] - C[:, j_idx]) # Shape: (K, num_pairs)

        # Dot product counts matches (+1) and mismatches (-1) across all pairs
        sum_signs = np.dot(sign_X, sign_C.T) # Shape: (N, K)

        # Apply the formula: 1 - (2 / m(m-1)) * sum_signs
        return 1 - (2 / (m * (m - 1))) * sum_signs

    else:
        raise ValueError(f"Unsupported metric: '{metric}'")

## `KNN` class

In [2]:
class KNN:
    def __init__(self, k=3, metric='euclidean', **kwargs):
        self.k = k
        self.metric = metric
        self.kwargs = kwargs
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X_test):
        # 1. Calculate distances from each test point to all training points.
        # X_test shape: (M, D), self.X_train shape: (N, D)
        # distances shape will be (M, N)
        distances = compute_distances(X_test, self.X_train, metric=self.metric, **self.kwargs)

        # 2. Get the indices of the 'k' smallest distances for each test point.
        # np.argsort returns the indices that would sort the array along axis 1.
        # We only need the first 'k' indices.
        k_indices = np.argsort(distances, axis=1)[:, :self.k]

        # 3. Retrieve the corresponding labels from the training set.
        k_nearest_labels = self.y_train[k_indices]

        # 4. Perform a majority vote for each test point.
        # We use Counter to find the most common label in each row of neighbors.
        predictions = [Counter(row).most_common(1)[0][0] for row in k_nearest_labels]

        return np.array(predictions)

## Test with artificial data for different distance metrics

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_classes=2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn_euclidean = KNN(k=5, metric='euclidean')
knn_euclidean.fit(X_train, y_train)
preds_euclidean = knn_euclidean.predict(X_test)
acc_euclidean = accuracy_score(y_test, preds_euclidean)

print(f"KNN Accuracy (Euclidean): {acc_euclidean * 100:.2f}%")

knn_manhattan = KNN(k=5, metric='manhattan')
knn_manhattan.fit(X_train, y_train)
preds_manhattan = knn_manhattan.predict(X_test)
acc_manhattan = accuracy_score(y_test, preds_manhattan)

print(f"KNN Accuracy (Manhattan): {acc_manhattan * 100:.2f}%")

knn_minkowski = KNN(k=5, metric='minkowsky', r=0.5)
knn_minkowski.fit(X_train, y_train)
preds_minkowski = knn_minkowski.predict(X_test)
acc_minkowski = accuracy_score(y_test, preds_minkowski)

print(f"KNN Accuracy (Minkowski r=0.5): {acc_minkowski * 100:.2f}%")

KNN Accuracy (Euclidean): 94.50%
KNN Accuracy (Manhattan): 94.00%
KNN Accuracy (Minkowski r=0.5): 93.00%
